In [ ]:
from tqdm import tqdm
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import statistics

In [ ]:
macs = {}

for chrom in tqdm(range(1, 23)):

    path = f'/mnt/project/DNM/twins/diffs (full)/chr{chrom}/'

    with open(f'{path}twins_unrelated_chr{chrom}.txt') as f:

        for row in f:

            row = row.strip()
            row = row.split(' ')

            diff_id = row[0]
            diff_mac = int(row[1])
            diff_obs = int(row[2])

            macs[diff_id] = (diff_mac, diff_obs)

## QC

In [ ]:
dp_lower_thre = 20
dp_upper_thre = 100

gq_thre = 30
p_thre = 0.05

In [ ]:
inds_diffs_qc = {}

path = f'/mnt/project/DNM/twins/QC (full)/'

for chrom in tqdm(range(1, 23)):
    for b in range(2):

        with open(f'{path}twins_qc_chr{chrom}_b{b}.txt') as f:

            for row in f:

                row = row.strip()
                row = row.split(' ')
                
                ind_idx = str(row[0])
                if ind_idx not in inds_diffs_qc:
                    inds_diffs_qc[ind_idx] = []
    
                pos = int(row[1].split(':')[1])
                ref = row[1].split(':')[2]
                alt = row[1].split(':')[3]
                
                mac, obs = macs[row[1]]
    
                ref_ad = int(row[2])
                alt_ad = int(row[3])
                dp = int(row[4])
                gq = int(row[5])
    
                twin_ref_ad = int(row[8])
                twin_alt_ad = int(row[9])
                twin_dp = int(row[10])
                twin_gq = int(row[11])
    
                if (dp >= dp_lower_thre and twin_dp >= dp_lower_thre and
                    gq >= gq_thre and twin_gq >= gq_thre and
                    dp <= dp_upper_thre and twin_alt_ad == 0):
    
                    p = binomtest(alt_ad, dp, 0.5).pvalue
    
                    if p > p_thre:
                        inds_diffs_qc[ind_idx].append((chrom, pos, ref, alt, mac, obs,
                                                       alt_ad, dp, gq, twin_alt_ad, twin_dp, twin_gq))

In [ ]:
cnts = []

for ind_idx in inds_diffs_qc:
    cnt = 0
    for diff in inds_diffs_qc[ind_idx]:
        if len(diff[2]) == 1 and len(diff[3]) == 1:
            cnt += 1
    cnts.append(cnt)
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per twin')
plt.show()

In [ ]:
cnts = []

for ind_idx in inds_diffs_qc:
    cnt = 0
    for diff in inds_diffs_qc[ind_idx]:
        if len(diff[2]) > 1 or len(diff[3]) > 1:
            cnt += 1
    cnts.append(cnt)
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of indel differences per twin')
plt.show()

# Record differences

In [ ]:
with open('twins_diffs_full_sigs.txt', 'w') as f:
    f.write('Project\tSample\tID\tGenome\tmut_type\tchrom\tpos_start\tpos_end\tref\talt\tType\n')
    for idx, ind in enumerate(inds_diffs_qc):
        for diff in inds_diffs_qc[ind]:
            if len(diff[2]) == 1 and len(diff[3]) == 1:
                f.write(f'.\t{idx}\t.\tGRCh38\tSNP\t{diff[0]}\t{diff[1]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t.\n')

In [ ]:
with open('twins_diffs_full_info.txt', 'w') as f:
    f.write('IID\tCHROM\tPOS\tREF\tALT\tMAC\tOBS\n')
    for ind in inds_diffs_qc:
        for diff in inds_diffs_qc[ind]:
            f.write(f'{ind}\t{diff[0]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t{diff[4]}\t{diff[5]}\n')

In [ ]:
with open('twins_diffs_full_macs.txt', 'w') as f:
    f.write('CHROM\tPOS\tREF\tALT\tMAC\tOBS\n')
    for ind in inds_diffs_qc:
        for diff in inds_diffs_qc[ind]:
            f.write(f'{diff[0]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t{diff[4]}\t{diff[5]}\n')